# Confronto semplice tra Batch, Mini-batch e Stochastic Gradient Descent

In questo notebook confrontiamo **3 varianti del gradient descent** su una funzione costo con **2 variabili** ($w_1, w_2$).
Obiettivi:
- capire la differenza pratica tra Batch GD, Mini-batch GD e SGD
- osservare come cambia la discesa della cost function
- visualizzare il percorso dei parametri su un contour plot 2D

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use("seaborn-v0_8")

## 1) Dataset sintetico e funzione costo

Usiamo una regressione lineare senza bias:

$$
\hat{y} = w_1 x_1 + w_2 x_2 = (x_1, \  x_2)\cdot (w_1, \ w_2)^T
$$

La cost function (MSE) dipende quindi da **due soli parametri**:

$$
C(w_1, w_2) = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2,
$$

dove $N=1$ per SGD, $N$ coincide con $N_{train}$ (numero di dati nel training set) per vanilla gradient descent, e $1<N<N_{train}$ per mini-batch.

Poi visualizzeremo il processo di training nel piano bidimensionale $(w_1, w_2)$.

In [2]:
# Generazione dati
n_samples = 500
X = np.random.randn(n_samples, 2)
true_w = np.array([3.0, -2.0])
noise = np.random.randn(n_samples)
y = X @ true_w + noise

# Funzioni utili
def predict(X, w):
    return X @ w

def cost_mse(X, y, w):
    err = predict(X, w) - y
    return np.mean(err ** 2)

def gradient_mse(X_batch, y_batch, w):
    err = X_batch @ w - y_batch
    return (2.0 / len(X_batch)) * (X_batch.T @ err)

print("Forma X:", X.shape)
print("Pesi reali:", true_w)

Forma X: (500, 2)
Pesi reali: [ 3. -2.]


## 2) Implementazione dei 3 tipi di Gradient Descent

- **Batch GD**: usa tutti i dati a ogni aggiornamento
- **Mini-batch GD**: usa piccoli gruppi di dati
- **Stochastic GD (SGD)**: usa un solo campione per volta

Per confrontarli in modo corretto, registriamo:
- traiettoria dei pesi $(w_1, w_2)$
- valore della cost function a ogni epoca (calcolato su tutto il dataset)

# Esercizio: 

1) Scrivere una funzione python (con anche altre funzioni ausiliare se necessario) che effettua il training del modello definito prima. La funzione calcola il gradiente e aggiorna i parametri tramite la regola di update ordinaria. Definisci tutti gli altri "iperparametri" che servono per questo scopo. La funzione deve accettare un parametro di input "mode" che gestisce i 3 tipi di gradient descent visti prima.

2) Mostrare il grafico del percorso seguito dai parametri aggiornati, epoca per epoca, in un piano 2d dei parametri $(w_1, w_2)$. Il grafico mostrerà 3 line plot, uno per ogni tipo di gradient descent.

3) Mostrare il grafico della Cost function come funzione dell'epoca di training. Il grafico mostrerà 3 line plot, uno per ogni tipo di gradient descent.

# PASSO 1

In [3]:
def train_gradiente_descent(X, y, w_init, train_w, mode="batch", lr = 0.1, epoche = 50, batch_size=32, seed=0):
    #mode = batch, minibatch, sgd
    rng = np.random.RandomState(seed)
    w = w_init.copy()
    N = len(X)

    peso_storico = [w.copy()]  # per la traiettoria dei pesi
    costo_storico = [cost_mse(X, y, w)] #è il costo iniziale su tutto il dataset

    for epoca in range(epoche):
        if mode == "batch":
            # per un solo update usando tutto il dataset
            grad = gradient_mse(X, y, w)
            w = w - lr * grad
        elif mode == "sgd":
            idx = rng.permutation(N)
            for i in idx:
                Xi, yi = X[i:i+1], y[i:i+1]
                grad = gradient_mse(Xi, yi, w)
                w = w - lr * grad
        elif mode == "minibatch":
            # un update per ogni mini batch
            idx = rng.permutation(N)
            for start in range(0, N, batch_size):
                batch_idx= idx[start:start+batch_size]
                Xb, yb = X[batch_idx], y[batch_idx]
                grad = gradient_mse(Xb, yb, w)
                w = w - lr*grad
        else:
            raise ValueError("il mode deve essere Batch, minibatch o sgd")
        peso_storico.append(w.copy())
        costo_storico.append(cost_mse(X, y, w))
        return np.array(peso_storico), np.array(costo_storico)
